In [ ]:
# ============================================================
# RetailPulse | Phase 9: Advanced Analytics
# Author: Naisha
# Date: June 2026
# Purpose: RFM Segmentation, Cohort Analysis, Forecasting
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Load cleaned tables
PROCESSED = "../data/processed/"
RAW = "../data/raw/"

orders = pd.read_csv(PROCESSED + "orders_clean.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

payments = pd.read_csv(PROCESSED + "payments_clean.csv")
customers = pd.read_csv(RAW + "olist_customers_dataset.csv")

print("✓ Orders:", orders.shape)
print("✓ Payments:", payments.shape)
print("✓ Customers:", customers.shape)

In [ ]:
# Step 1 — Join orders, payments, customers
rfm_base = (
    orders
    .merge(payments, on='order_id')
    .merge(customers, on='customer_id')
)

# Only delivered orders
rfm_base = rfm_base[rfm_base['order_status'] == 'delivered']

print("✓ Base table shape:", rfm_base.shape)
print("✓ Columns:", rfm_base.columns.tolist())

In [ ]:
# Step 2a — Find last order date per customer
last_order = rfm_base.groupby('customer_unique_id')['order_purchase_timestamp'].max()
last_order = last_order.reset_index()
last_order.columns = ['customer_unique_id', 'last_order_date']

# Step 2b — Find order count per customer
order_count = rfm_base.groupby('customer_unique_id')['order_id'].nunique()
order_count = order_count.reset_index()
order_count.columns = ['customer_unique_id', 'frequency']

# Step 2c — Find total spend per customer
total_spend = rfm_base.groupby('customer_unique_id')['payment_value'].sum()
total_spend = total_spend.reset_index()
total_spend.columns = ['customer_unique_id', 'monetary']

print("✓ Last order done")
print("✓ Frequency done")
print("✓ Monetary done")

In [ ]:
# Step 3 — Combine into one RFM table
rfm = last_order.merge(order_count, on='customer_unique_id')
rfm = rfm.merge(total_spend, on='customer_unique_id')

# Calculate recency in days
reference_date = orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
rfm['recency'] = (reference_date - rfm['last_order_date']).dt.days

# Drop last_order_date — no longer needed
rfm = rfm.drop('last_order_date', axis=1)

print("✓ RFM table complete")
print(rfm.shape)
print(rfm.head())

In [ ]:
# Step 4 — Score each customer 1 to 4
# Recency: lower days = better = higher score
rfm['r_score'] = pd.qcut(rfm['recency'], q=4, labels=[4, 3, 2, 1])

# Frequency: higher = better = higher score
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 
                          q=4, labels=[1, 2, 3, 4])

# Monetary: higher = better = higher score
rfm['m_score'] = pd.qcut(rfm['monetary'], q=4, labels=[1, 2, 3, 4])

print("✓ RFM scores added")
print(rfm.head())

In [ ]:
# Step 5 — Create RFM segment labels
rfm['r_score'] = rfm['r_score'].astype(int)
rfm['f_score'] = rfm['f_score'].astype(int)
rfm['m_score'] = rfm['m_score'].astype(int)

# Combined RFM score
rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

# Segment labels
def assign_segment(row):
    r = row['r_score']
    f = row['f_score']
    m = row['m_score']
    
    if r >= 3 and f >= 3:
        return 'Champions'
    elif r >= 3 and f >= 2:
        return 'Loyal Customers'
    elif r >= 3 and f == 1:
        return 'New Customers'
    elif r == 2 and f >= 2:
        return 'At Risk'
    elif r <= 2 and f >= 3:
        return 'Cannot Lose Them'
    else:
        return 'Lost'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

print("✓ Segments assigned")
print(rfm['segment'].value_counts())

In [ ]:
# Step 6 — Visualize segments
plt.figure(figsize=(8,5))
segment_counts = rfm['segment'].value_counts()
colors = ['green', 'purple', 'red', 'yellow', 'blue', 'orange']

segment_counts.plot(kind='bar', color=colors, edgecolor='none')
plt.title('Customer Segments — RFM Analysis', fontsize=12)
plt.xlabel('Segment')
plt.ylabel('Number of Customers')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Save RFM results
rfm.to_csv('../data/processed/rfm_segments.csv', index=False)
print("✓ RFM segments saved")

## RFM Segmentation Results

### Segment Breakdown:
- **Champions (25%)** — buy recently and often → reward and retain
- **Lost (19%)** — inactive customers → last chance campaigns
- **At Risk (19%)** — slipping away → urgent win-back needed
- **New Customers (12%)** — recent but once → nurture to loyalty
- **Loyal Customers (12%)** — regular buyers → upsell opportunities
- **Cannot Lose Them (12%)** — high value going quiet → immediate re-engagement

### Key Business Insight:
At Risk + Cannot Lose Them = 28,980 customers (31%)
are slipping away. A targeted email campaign for these
two segments could recover significant lost revenue.

### Marketing Actions:
- Champions → VIP programme, early access to new products
- At Risk → "We miss you" email + 15% discount
- Cannot Lose Them → Personal outreach + exclusive offer
- Lost → Final win-back email, then remove from active list